In [1]:
# ============================================================
# Notebook : 01_data_preparation.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Load the parsed sensor table, build the binary risk target and
#            per-household sliding windows, and define a leakage-free
#            household-level Group 5-fold split. All arrays are cached under
#            data/processed/ for the downstream notebooks.
# Note     : Anonymous repository for double-blind review.
# ============================================================

import os, random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# ── Config ───────────────────────────────────────────────────
SEED    = 42
random.seed(SEED); np.random.seed(SEED)
WINDOW  = 6          # look-back window (6 x 10 min = 60 min)
STRIDE  = 2          # window stride (=2 for CPU efficiency; set 1 for max samples)
N_FOLDS = 5          # household-level Group K-fold
FEAT_COLS = ['Activity_IR', 'HeartRate', 'BreathRate', 'StressIndex',
             'Temperature', 'Humidity', 'CO2', 'TVOC']   # 8 sensors used as nodes
EM_IDX = [0, 4, 5, 6, 7]      # environmental sensors
SM_IDX = [1, 2, 3]            # physiological sensors
# Binary risk target: 위험(danger)+주의(caution) -> 1 ; everything else -> 0
LABEL_MAP = {'위험': 1, '주의': 1, '수면': 0, '식사': 0, '외출': 0, '기타': 0}

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"
TAB_DIR  = ROOT / "results" / "tables"
PROC_DIR.mkdir(parents=True, exist_ok=True); TAB_DIR.mkdir(parents=True, exist_ok=True)

# ── Load parsed table ────────────────────────────────────────
df = pd.read_pickle(PROC_DIR / "df_new.pkl")
df['label'] = df['Estimation'].map(LABEL_MAP)
df = df[df['label'].notna()].copy()
df['label'] = df['label'].astype(int)
df.sort_values(['HH_ID', 'Timestamp'], inplace=True)
print(f"Loaded table : {df.shape}  | households : {df['HH_ID'].nunique()}")
print(f"Positive (risk) rate : {df['label'].mean()*100:.1f}%")

# ── Build per-household sliding windows (raw, UNSCALED) ───────
# Scaling is deliberately left to the model notebooks so it can be fit on the
# training households of each fold only (no leakage).
X_list, y_list, hh_list_s = [], [], []
feat_mat = df[FEAT_COLS].values.astype(np.float32)
for hh, grp in df.groupby('HH_ID'):
    pos  = df.index.get_indexer(grp.index)
    f    = feat_mat[pos]                       # (T, 8)
    labs = grp['label'].values.astype(np.int64)
    for st in range(0, len(f) - WINDOW, STRIDE):
        x = f[st:st + WINDOW]                  # (W, 8)
        if np.isnan(x).any():
            continue
        X_list.append(x.T)                     # (8, W)  -> node x time
        y_list.append(labs[st + WINDOW])
        hh_list_s.append(hh)

X  = np.asarray(X_list, dtype=np.float32)      # (N, 8, W)
y  = np.asarray(y_list, dtype=np.int64)        # (N,)
hh = np.asarray(hh_list_s)                     # (N,)
print(f"\nWindows : X={X.shape}  y={y.shape}  (stride={STRIDE})")
print(f"Window positive rate : {y.mean()*100:.1f}%")

# ── Household-level Group 5-fold split ───────────────────────
hh_unique = np.array(sorted(df['HH_ID'].unique()))
gkf = GroupKFold(n_splits=N_FOLDS)
sample_fold = np.full(len(y), -1, dtype=np.int64)   # test-fold id per sample
hh_fold = {}
for fold, (_, te_idx) in enumerate(gkf.split(X, y, groups=hh)):
    sample_fold[te_idx] = fold
    for h in np.unique(hh[te_idx]):
        hh_fold[h] = fold
assert (sample_fold >= 0).all()

fold_tbl = (pd.DataFrame({'HH_ID': hh_unique,
                          'fold': [hh_fold[h] for h in hh_unique]})
            .sort_values(['fold', 'HH_ID']))
print("\nHousehold -> test fold assignment:")
for f in range(N_FOLDS):
    members = fold_tbl.loc[fold_tbl.fold == f, 'HH_ID'].tolist()
    print(f"  fold {f}: {len(members)} households  {members}")

# ── Class weight (inverse frequency, computed later per fold too) ─
pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"\nGlobal pos_weight (neg/pos) : {pos_weight:.3f}")

# ── Save caches ──────────────────────────────────────────────
np.savez_compressed(PROC_DIR / "windows.npz",
                    X=X, y=y, hh=hh, sample_fold=sample_fold,
                    feat_cols=np.array(FEAT_COLS), window=WINDOW,
                    em_idx=np.array(EM_IDX), sm_idx=np.array(SM_IDX))
fold_tbl.to_csv(PROC_DIR / "household_folds.csv", index=False)
fold_tbl.to_csv(TAB_DIR / "household_folds.csv", index=False)

# Per-household summary table (for the paper appendix)
summary = (df.groupby('HH_ID')
             .agg(n_rows=('label', 'size'), risk_rate=('label', 'mean'),
                  age=('Age', 'first'), gender=('Gender', 'first'))
             .reset_index())
summary['fold'] = summary['HH_ID'].map(hh_fold)
summary['risk_rate'] = (summary['risk_rate'] * 100).round(1)
summary.to_csv(TAB_DIR / "household_summary.csv", index=False)

print(f"\nSaved -> {PROC_DIR/'windows.npz'}")
print(f"Saved -> {PROC_DIR/'household_folds.csv'}")
print(f"Saved -> {TAB_DIR/'household_summary.csv'}")
print("\n01_data_preparation complete.")


Loaded table : (109819, 28)  | households : 27
Positive (risk) rate : 21.9%



Windows : X=(54836, 8, 6)  y=(54836,)  (stride=2)
Window positive rate : 21.8%

Household -> test fold assignment:
  fold 0: 5 households  ['A0008', 'A0012', 'A0017', 'A0025', 'A0028']
  fold 1: 7 households  ['A0003', 'A0006', 'A0007', 'A0013', 'A0019', 'A0021', 'A0029']
  fold 2: 5 households  ['A0009', 'A0016', 'A0018', 'A0022', 'A0026']
  fold 3: 5 households  ['A0010', 'A0014', 'A0015', 'A0024', 'A0027']
  fold 4: 5 households  ['A0002', 'A0004', 'A0005', 'A0023', 'A0030']

Global pos_weight (neg/pos) : 3.586



Saved -> /home/claude/causal-multimodal-elderly-risk/data/processed/windows.npz
Saved -> /home/claude/causal-multimodal-elderly-risk/data/processed/household_folds.csv
Saved -> /home/claude/causal-multimodal-elderly-risk/results/tables/household_summary.csv

01_data_preparation complete.
